In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

df = spark.read.table("company_risk_intelligence_platform.bronze.yf_stock")
df.display()

In [0]:
df.printSchema()

#Step 1: Create Transformation Function

For stocks, our Silver responsibilities are:

- Extract ticker from file_path
- Convert date to proper date type
- Remove technical column file_path
- Remove duplicates
- Reorder columns

In [0]:
from pyspark.sql.functions import (
    col,
    regexp_extract,
    to_date
)

def transform_stock(df):

    df = (
        df
        .withColumn(
            "ticker",
            regexp_extract(
                col("file_path"),
                r"/stock/\d{4}/\d{2}/\d{2}/([^/]+)/",
                1
            )
        )
        .withColumn(
            "trading_date",
            to_date(col("date"))
        )
        .drop("date", "file_path")
    )

    # Business key = ticker + trading_date
    df = df.dropDuplicates(["ticker", "trading_date"])

    df = df.select(
        "ticker",
        "trading_date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "dividends",
        "stock_splits",
        "ingestion_ts"
    )

    return df

#Step 2: SCD1 Merge Function

In [0]:
def scd_merge_table(spark, source_table, target_table, business_key):

    if not spark.catalog.tableExists(target_table):

        print("First Load: Creating Silver Table", target_table)

        source_table.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

        print("Table Created")

    else:

        print("Incremental Load: Performing SCD Type 1 Merge")

        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )

        delta_table.alias("target") \
            .merge(
                source_table.alias("source"),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()

        print("Merge Successfully Completed")

### SCD Type 1 Merge Function

This function loads data into a Delta table using an SCD Type 1 (overwrite) strategy. During the initial load, it creates the target Delta table if it does not already exist. For subsequent loads, it performs a Delta Lake merge using the specified business key(s), updating existing records when a match is found and inserting new records when no match exists. This approach ensures that the target table always contains the most recent version of each record while maintaining a single row per business key.


# Step 3: Main Logic
# 


In [0]:
########################################
######## MAIN LOGIC ####################
########################################

source_table = "company_risk_intelligence_platform.bronze.yf_stock"
target_table = "company_risk_intelligence_platform.silver.yf_stock"

business_key = ["ticker", "trading_date"]

df = spark.read.table(source_table)

source_table = transform_stock(df)

scd_merge_table(
    spark,
    source_table,
    target_table,
    business_key
)

### Processing Steps

1. Read raw stock data from the Bronze Delta table.
2. Apply data cleansing and standardization through the `transform_stock()` transformation function.
3. Define `ticker` and `trading_date` as the composite business key.
4. Load the transformed data into the Silver Delta table using an SCD Type 1 merge process.
5. Update existing records and insert new records to maintain a current, deduplicated dataset.


In [0]:
df = spark.read.table("company_risk_intelligence_platform.silver.yf_stock")
df.display()